# Sentinel‑2 Panel Cropper — whitespace‑free version

This notebook scans **`input_panels/`** for files that end with `_panel.png`,
extracts the six square tiles (RGB, Cloud, Land, Solid ice, Light ice, Overlay),
trims away any white border around each tile, and saves them to  
**`public/data/panels‑crops/`** as individual JPEGs named

```
YYYY-MM-DD_rgb.jpg
YYYY-MM-DD_cloud.jpg
…
```

*You only need to run it when new panels are added.*

In [1]:
# ⬇️ Install Pillow if you haven't already
# !pip install pillow

In [2]:
from pathlib import Path
import re
from PIL import Image
import numpy as np

# -------- CONFIG ---------------------------------------------------
SRC_DIR = Path("out/imgtestNEW")                # where *_panel.png files live
DST_DIR = Path("out/panels-crops")    # where cropped tiles go
DST_DIR.mkdir(parents=True, exist_ok=True)

ORDER = ["rgb","cloud","land",
         "solid","light","overlay"]

DATE_RX = re.compile(r"(\d{4})(\d{2})(\d{2})")


In [ ]:
def date_from_name(fname:str):
    """Extract YYYY-MM-DD from filename (first 8-digit run)."""
    m = DATE_RX.search(fname)
    return f"{m[1]}-{m[2]}-{m[3]}" if m else None

import numpy as np
from PIL import Image

def tight_crop(tile: Image.Image, thresh:int = 245, min_frac:float = 0.05
               ) -> Image.Image:
    """
    Remove surrounding whitespace *and* the label strip.

    • `thresh`  – anything < thresh is considered “content”
    • `min_frac`– minimum fraction of content pixels a row/col must have
                  to be kept (0.05 = 5 %)
    """
    arr = np.array(tile.convert("L"))        # grayscale 0‥255
    mask = arr < thresh                      # True = content
    if not mask.any():
        return tile                          # blank tile

    # fraction of content per row / column
    row_frac = mask.mean(axis=1)
    col_frac = mask.mean(axis=0)

    # rows/cols where content fraction exceeds the threshold
    rows = np.where(row_frac > min_frac)[0]
    cols = np.where(col_frac > min_frac)[0]

    top, bottom = rows[0], rows[-1] + 1
    left, right = cols[0], cols[-1] + 1

    return tile.crop((left, top, right, bottom))

def process_panels(src:Path=SRC_DIR, dst:Path=DST_DIR):
    processed = 0
    for fp in src.glob("*panel.png"):
        date = date_from_name(fp.name)
        if not date:
            print("skip", fp.name)
            continue

        im = Image.open(fp)
        W, H = im.size
        cw, ch = W//3, H//2
        side = min(cw, ch)

        for idx, tag in enumerate(ORDER):
            col, row = idx % 3, idx // 3
            left, top = col*cw, row*ch
            crop = im.crop((left, top, left+side, top+side))
            crop = tight_crop(crop)

            if crop.mode in ("RGBA","P"):
                crop = crop.convert("RGB")

            out = dst / f"{date}_{tag}.jpg"
            crop.save(out, quality=92)
        processed += 1
    print(f"✓  {processed} mosaics processed → {dst}")

In [4]:
process_panels()

✓  907 mosaics processed → out/panels-crops
